# Fine-tuning QLoRA - MedAssist (Colab)

Espelho do script `src/medassist/finetune/train.py` para execucao em GPU gratuita (Colab/Kaggle). Roda **fora** da VPS de producao (que nao tem GPU) - o artefato final (GGUF) e que vai para producao via Ollama.

Celulas: **install** -> **mount** -> **train** -> **export** -> **download**.

## 1. Install

In [ ]:
!pip install -q unsloth trl peft bitsandbytes datasets
!pip install -q "git+https://github.com/huggingface/transformers.git"

## 2. Mount (Google Drive - dataset e saida dos adaptadores)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_PATH = '/content/drive/MyDrive/medassist/train.jsonl'  # copiar de data/processed/train.jsonl
OUT_DIR = '/content/drive/MyDrive/medassist/adapters'

## 3. Train (QLoRA via Unsloth)

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset

BASE_MODEL = 'unsloth/Llama-3.2-3B-Instruct'
R, ALPHA, LR, EPOCHS, MAX_SEQ_LEN = 16, 32, 2e-4, 2, 2048

modelo, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True,
)
modelo = FastLanguageModel.get_peft_model(
    modelo, r=R, lora_alpha=ALPHA,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',
)

dataset = load_dataset('json', data_files=DATASET_PATH, split='train')

config = SFTConfig(
    output_dir=OUT_DIR, per_device_train_batch_size=2, gradient_accumulation_steps=4,
    num_train_epochs=EPOCHS, learning_rate=LR, logging_steps=10,
    save_strategy='epoch', max_seq_length=MAX_SEQ_LEN,
)
trainer = SFTTrainer(model=modelo, args=config, train_dataset=dataset, tokenizer=tokenizer)
trainer.train()

modelo.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print(f'Adaptadores salvos em {OUT_DIR}')

## 4. Export (merge + GGUF)

In [ ]:
modelo_merged = FastLanguageModel.for_inference(modelo)
modelo.save_pretrained_merged('/content/merged', tokenizer, save_method='merged_16bit')

!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt
!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged --outfile /content/medassist-f16.gguf
!cd /content/llama.cpp && cmake -B build && cmake --build build --target llama-quantize -j
!/content/llama.cpp/build/bin/llama-quantize /content/medassist-f16.gguf /content/medassist-q4_k_m.gguf Q4_K_M

## 5. Download

In [ ]:
from google.colab import files
files.download('/content/medassist-q4_k_m.gguf')

# Ou copiar para o Drive e baixar depois com scp/rsync para a VPS em ./models/